# TEST CELLPOSE ON TRAIN SET

In [1]:
import os
import argparse
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from cellpose.models import CellposeModel

from metric import score
from generate_submission import build_submission
from generate_train_submission import build_submission as one_submission

In [ ]:
def get_stats(x):
    print(f"MEAN: {np.mean(x)}")
    print(f"MEDIAN: {np.median(x)}")
    print(f"MIN: {np.min(x)}")
    print(f"MAX: {np.max(x)}")

def load_dax(filepath, height=2048, width=2048):
    """Load a .dax raw image file. Raw uint16 binary, no header."""
    raw = np.fromfile(filepath, dtype=np.uint16)
    n_frames = len(raw) // (height * width)
    return raw.reshape(n_frames, height, width)

In [ ]:
epi_stack = load_dax('FOV_001/Epi-750s5-635s5-545s1-473s5-408s5_001.dax')
print(f'Epi stack shape: {epi_stack.shape}  (frames, height, width)')

In [6]:
z_plane = 2  # middle z-plane
dapi = epi_stack[6 + z_plane * 5]   # frame 16 for z2
polyt = epi_stack[5 + z_plane * 5]  # frame 15 for z2

In [7]:
# Model evaluation

# Cellpose v4+: use CellposeModel (not models.Cellpose)
model = CellposeModel(model_type='nuclei', gpu=True)

# eval() returns 3 values: masks, flows, styles
masks, flows, styles = model.eval(dapi, diameter=30, channels=[0, 0])

print(f'Segmentation complete!')
print(f'Mask shape: {masks.shape}')
print(f'Number of cells found: {masks.max()}')

model_type argument is not used in v4.0.1+. Ignoring this argument...
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


Segmentation complete!
Mask shape: (2048, 2048)
Number of cells found: 218


In [8]:
# Save masks to file
np.save('FOV_001_mask.npy', masks)

In [9]:
!python3 generate_train_submission.py --mask_A FOV_001_mask.npy

Loading test_spots.csv...
  224,495 spots across 4 FOVs

Loading masks...
  FOV_001: shape=(2048, 2048), dtype=uint16, 218 cells

Building submission...

Submission written to submission.csv
  Rows: 224,495
  Columns: ['spot_id', 'fov', 'cluster_id']
  Unique clusters: 1
  Background spots: 224,495

Upload this file to Kaggle for scoring.


## Load Spots Train CSV

In [10]:
spots_train_df = pd.read_csv('spots_train.csv')
# Check to see the range of values
spots_train_df['global_x'].max(), spots_train_df['global_x'].min(), spots_train_df['global_x'].median(), spots_train_df['global_y'].max(), spots_train_df['global_y'].min(), spots_train_df['global_y'].median()

(np.float64(5798.334),
 np.float64(-2402.6602),
 np.float64(675.7866),
 np.float64(2647.3037),
 np.float64(-1953.6913),
 np.float64(746.13824))

In [11]:
spots_train_df = spots_train_df[spots_train_df['fov'] == 'FOV_001']

In [12]:
spots_train_df['fov'].value_counts()

fov
FOV_001    117893
Name: count, dtype: int64

In [13]:
import torch
import gc
# 1. Delete large objects
# del model 
# 2. Force garbage collection
gc.collect()
# 3. Clear the PyTorch cache
torch.cuda.empty_cache()

## Load Test Set

In [14]:
test_spots_df = pd.read_csv('test_spots.csv')
test_spots_df
# We need to recreate the spot_id column

,spot_id,fov,image_row,image_col,global_x,global_y,global_z,target_gene
0,spot_0,FOV_B,1899,511,-1797.5458,90.935470,0.0,Tfap2d
1,spot_1,FOV_B,812,140,-1679.0660,50.451340,0.0,Cyp26b1
2,spot_2,FOV_B,1913,190,-1799.0302,55.889180,0.0,Cyp26b1
3,spot_3,FOV_B,1275,734,-1729.5323,115.190510,0.0,Cyp26b1
4,spot_4,FOV_B,1297,765,-1731.9692,118.579680,0.0,Cyp26b1
...,...,...,...,...,...,...,...,...
224490,spot_224490,FOV_C,1516,612,-1755.7604,-98.107956,4.0,Itih5
224491,spot_224491,FOV_C,1566,670,-1761.2257,-91.738770,4.0,Itih5
224492,spot_224492,FOV_C,160,1448,-1607.9718,-7.027220,4.0,Itih5
224493,spot_224493,FOV_C,1366,1686,-1739.4400,18.961140,4.0,Itih5


## Generate train submission

In [15]:
from generate_train_submission import

SyntaxError: invalid syntax (504300530.py, line 1)

## CALCULATE SCORE FROM 
- cell_boundaries_train.csv
- spots_train.csv


In [ ]:
from tqdm import tqdm

from shapely.geometry import Point, Polygon

poly = Polygon([(0, 0), (1, 0), (1, 1), (0, 1)])
point = Point(0.5, 0.5)
print(poly.contains(point))  # Output: True

In [ ]:
# Get cell boundries and convert from a string to a list

def parse_float_list(text):
    if isinstance(text, str):
        return np.fromstring(text, sep=',').tolist()
    return None

cell_boundaries_train_df = pd.read_csv('cell_boundaries_train.csv')
cell_boundaries_train_df.iloc[:, 1:] = cell_boundaries_train_df.iloc[:, 1:].applymap(parse_float_list)

# spots_train_df = pd.read_csv('spots_train.csv')
# spots_train_df = spots_train_df[spots_train_df['fov'].isin(['FOV_019', 'FOV_001'])]
# spots_train_df = spots_train_df[spots_train_df['fov'].isin(['FOV_019', 'FOV_001'])]
spots_train_df = pd.read_csv('spots_train.csv')
spots_train_df = spots_train_df[spots_train_df['fov'] == 'FOV_001']

# For each FOV, for each Z, Mask -> split into individiual pixels
# spots = spots_train_df[['fov', 'image_row', 'image_col', 'global_z']]
spots = spots_train_df
spots.head(4)

In [ ]:
cell_boundaries_train_df.head(4)

In [ ]:
def get_first(x):
    if x:
        return x[0]
    return None
x_boundry_test = cell_boundaries_train_df['boundaryX_z2'].transform(get_first)
cell_boundaries_train_df.shape, x_boundry_test.min(), x_boundry_test.max(), x_boundry_test.median()


In [ ]:
# This shows that some cells do not span across entire z-plane.
# cell_boundaries_train_df[cell_boundaries_train_df.iloc[:, 1:].isna().sum(axis=1) > 0]
# cell_boundaries_train_df[cell_boundaries_train_df.iloc[:, 1:].isna().sum(axis=1) == 8]

# All cells listed have at least 1 z plane
cell_boundaries_train_df[cell_boundaries_train_df.iloc[:, 1:].isna().sum(axis=1) == 10]

# All Cell IDs exist
# cell_boundaries_train_df['Unnamed: 0'].isna().sum()

In [ ]:
spots.head()

In [ ]:
submission_df = []
# WE ITERATE ACROSS ALL SPOTS DETECTED (MASK) FIRST 
for s in tqdm(range(spots.shape[0])):
    spot_row = spots.iloc[s, :]
    z_level = int(spot_row['global_z'])
    submission_row = {
        'spot_id' : s,
        'fov' : spot_row['fov'],
        'gt_cluster_id' : 'background'
    }

    for c in tqdm(range(cell_boundaries_train_df.shape[0]), leave=False):
        cell_row = cell_boundaries_train_df.loc[c, ["Unnamed: 0", f"boundaryX_z{z_level}", f"boundaryY_z{z_level}"]]

        # print(cell_row[f"boundaryX_z{z_level}"])
        # print(cell_row)
        # print(cell_row.isna().sum() != 0)

        # if there is an NA, then the cell is not on this z-plane, skip
        if cell_row.isna().sum() != 0:
            break

        cell_x = [float(x) for x in cell_row[f"boundaryX_z{z_level}"].split(',')]
        cell_y = [float(x) for x in cell_row[f"boundaryY_z{z_level}"].split(',')]

        if ((min(cell_x) <= spot_row['image_col'] <= max(cell_x)) and 
            (min(cell_y) <= spot_row['image_row'] <= max(cell_y))):
            submission_row['gt_cluster_id'] = cell_row['"Unnamed: 0"']
            break

    submission_df.append(submission_row)

submission_df = pd.DataFrame(submission_df)

In [ ]:
submission_df.head()